# Causal OT — Visualization Notebook

Edit this notebook to iterate on plots. Each section loads data and produces figures.
Results CSVs are in the `results/` directory.

C1: causal OT stability under anisotropic deformations (simulation).
C2: recourse effort — ambient vs latent (5-fold CV on real datasets).
C3: real-data validation (ecoli70 BN simulation + PCA conjugacy deviation).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import ast

RESULTS_DIR = Path.cwd() / 'results'

# ── Tweaks you can edit ──────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (7, 4.5),
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})

DATASET_COLORS = {
    'adult': '#1f77b4',
    'german': '#ff7f0e',
    'compas': '#2ca02c',
    'lsac': '#d62728',
    'credit_default': '#9467bd',
    'heart_disease': '#8c564b',
    'saheart': '#e377c2',
    'student': '#7f7f7f',
    'acsincome': '#bcbd22',
    'communities': '#17becf',
    'hmda': '#aec7e8',
}

DATASET_LABELS = {
    'adult': 'Adult',
    'german': 'German Credit',
    'compas': 'COMPAS',
    'lsac': 'LSAC',
    'credit_default': 'Credit Default',
    'heart_disease': 'Heart Disease',
    'saheart': 'SA Heart',
    'student': 'Student',
    'acsincome': 'ACSIncome',
    'communities': 'Communities',
    'hmda': 'HMDA',
}

# ── Demographic group labels for theta in F2 ─────────────────
# Indices correspond to theta_0, theta_1, ... from the solver output.
# For multi-class attributes, leave as None (falls back to 'Group i').
GROUP_LABELS = {
    'german': ['Female', 'Male'],
    'adult': ['Female', 'Male'],
    'compas': ['African-American', 'Asian', 'Caucasian', 'Hispanic', 'Native American', 'Other'],
    'lsac': ['Non-white', 'White'],
    'credit_default': ['Female', 'Male'],
    'saheart': ['Absent fam.', 'Present fam.'],
    'student': ['Female', 'Male'],
    'communities': ['Non-white maj.', 'White maj.'],
    'heart_disease': ['Female', 'Male'],
    'acsincome': ['Female', 'Male'],
}


def load_csv(name):
    path = RESULTS_DIR / f'{name}.csv'
    if not path.exists():
        print(f'  WARNING: {path} not found')
        return pd.DataFrame()
    return pd.read_csv(path)


def savefig(fig, name):
    path = RESULTS_DIR / f'{name}.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved {path}')
    plt.show()


def parse_m_z(mz_str):
    try:
        return ast.literal_eval(mz_str) if isinstance(mz_str, str) else {}
    except:
        return {}

---
## C1 — Causal OT Stability Under Anisotropic Deformations

Simulation: X = M·U with M = Q + ε·N, N nilpotent.
Tests T_X vs M ∘ T_U ∘ M⁻¹ as ε varies.
Tracks anisotropy ratio κ = Λ/λ vs integral error.
50 simulations per ε, compact uniform ellipsoid support.

In [ ]:
df1_2d = load_csv('c1_causality_2d')
df1_5d = load_csv('c1_causality_5d')


if not df1_2d.empty or not df1_5d.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for label, df, ax, offset in [('2D', df1_2d, ax1, 1), ('5D', df1_5d, ax2, 0)]:
        if df.empty:
            ax.set_title(f'{label} — no data')
            continue
        df = df[df['c'] > 0.01]
        eps = df['c'].values
        kappa = df['kappa'].values - 1
        boundx = df['bound_X']
        boundu = df['bound_U']
        ax.plot(eps, df['integral_mean'].values, 'o-', color='#1f77b4',
                label=r'$\|T_X - \tilde{T}_X\|^2$')
        ax.plot(eps, df['u_discrepancy_mean'].values, 's-', color='#2ca02c',
                label=r'$\|T_U - \tilde{T}_U\|^2$')
        ax.plot(eps, boundx, '--', color='#1f77b4', alpha=0.6,
                  label=r'Bound in $X$')
        ax.plot(eps, boundu, '--', color='#2ca02c', alpha=0.6,
                  label=r'Bound in $U$')
        ax.set_ylabel(r'Transport cost', size=15)
        ax.set_xlabel(r'$c$', size=15)
        ax.set_title(f'{label} Conjugacy deviation (X vs U)', size=20)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.grid(True, which='both', alpha=0.3)
        lines1, labels1 = ax.get_legend_handles_labels()
        ax.legend(lines1, labels1, fontsize=15)
    fig.tight_layout()
    savefig(fig, 'c1_dual_discrepancy')
    plt.close(fig)
else:
    print('E1 results not found - run c1_causality.py first')

In [ ]:
# C1 — Log-log: integral & u\_discrepancy vs kappa (anisotropy ratio), estimate power
if not df1_2d.empty or not df1_5d.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for label, df, ax, pos in [('2D', df1_2d, ax1, {'X': (1e0, 2e1), 'U': (1e1, 1e-1)}), ('5D', df1_5d, ax2, {'X': (1e1, 1e2), 'U': (1e2, 1e0)})]:
        if df.empty:
            ax.set_title(f'{label} — no data')
            continue
        kappa_raw = df['kappa'].values - 1

        # --- integral (X-space discrepancy) fit ---
        mu = df['integral_mean'].values
        sd = df['integral_std'].values
        mask_i = np.logical_and(~np.isnan(kappa_raw), ~np.isnan(mu))
        ki = kappa_raw[mask_i]; mi = mu[mask_i]; sdi = sd[mask_i]
        valid_i = ki > 1e-17
        log_ki = np.log(ki[valid_i])
        log_mi = np.log(mi[valid_i])
        ci = np.polyfit(log_ki, log_mi, 1)
        p_i, logC_i = ci[0], ci[1]
        fit_i = np.exp(logC_i + p_i * log_ki)

        # --- u_discrepancy (U-space discrepancy) fit ---
        ud = df['u_discrepancy_mean'].values
        ud_sd = df['u_discrepancy_std'].values
        mask_u = np.logical_and(~np.isnan(kappa_raw), ~np.isnan(ud))
        ku = kappa_raw[mask_u]; udu = ud[mask_u]; ud_sdu = ud_sd[mask_u]
        valid_u = ku > 1e-10
        log_ku = np.log(ku[valid_u])
        log_udu = np.log(udu[valid_u])
        cu = np.polyfit(log_ku, log_udu, 1)
        p_u, logC_u = cu[0], cu[1]
        fit_u = np.exp(logC_u + p_u * log_ku)

        ax.errorbar(ki[valid_i], mi[valid_i], yerr=sdi[valid_i], fmt='o', capsize=3,
                    color='#1f77b4', label=r'$\|T_X - \tilde{T}_X\|^2$', alpha=0.9)
        line_i, = ax.plot(ki[valid_i], fit_i, '--', color='#1f77b4', alpha=0.9, linewidth=2)
        ax.text(pos['X'][0], pos['X'][1],
                rf'${np.exp(logC_i):.2e} \kappa^{{{p_i:.2f}}}$', color='#1f77b4', fontsize=15)

        ax.errorbar(ku[valid_u], udu[valid_u], yerr=ud_sdu[valid_u], fmt='s', capsize=3,
                    color='#2ca02c', label=r'$\|T_U - \tilde{T}_U\|^2$', alpha=0.9)
        line_u, = ax.plot(ku[valid_u], fit_u, '--', color='#2ca02c', alpha=0.9, linewidth=2)
        ax.text(pos['U'][0], pos['U'][1],
                rf'${np.exp(logC_u):.2e} \kappa^{{{p_u:.2f}}}$', color='#2ca02c', fontsize=15)

        ax.set_xlabel(r'$\kappa = \Lambda/\lambda - 1$', size=15)
        ax.set_ylabel('Transport cost', size=15)
        ax.set_title(rf'{label} Causal simulation', size=20)
        ax.legend(fontsize=15)
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    savefig(fig, 'c1_loglog_scaling')
    plt.close(fig)
else:
    print('E1 results not found - run c1_causality.py first')

---
## C2 — The Cost of Causal Ignorance: Latent Intervention Effort

Compares Standard OT vs Causal OT on 4 datasets.
Key metric: True Intervention Effort (latent L2) vs Apparent Cost (ambient L2).
The anisotropy ratio κ measures how skewed the causal graph is.

In [ ]:
# C2 — Thermodynamics of Recourse: ambient vs latent, and kappa penalty
df15 = load_csv('c2_recourse_causal')
df_ecoli = load_csv('c3_causality_ecoli70')
if not df_ecoli.empty:
    df15 = pd.concat([df15, df_ecoli], ignore_index=True)

# C2 — LaTeX table with CV uncertainty (mean \pm std across folds)
if not df15.empty:
    has_fold = 'fold' in df15.columns
    if has_fold:
        rows_list = []
        for ds_name in sorted(df15['dataset'].unique()):
            ds_df = df15[df15['dataset'] == ds_name]
            for method in ['Standard OT', 'Causal OT']:
                m_df = ds_df[[
#                    f'{method}_validity',
                    f'{method}_ambient_cost',
                    f'{method}_latent_effort'
                    ]].dropna()
                if m_df.empty:
                    continue
                rows_list.append({
                    'dataset': ds_name,
                    'kappa_mean': ds_df['kappa'].mean(),
                    'kappa_std': ds_df['kappa'].std(),
                    'method': method,
#                    'validity_mean': m_df[f'{method}_validity'].mean(),
#                    'validity_std': m_df[f'{method}_validity'].std(),
                    'ambient_mean': m_df[f'{method}_ambient_cost'].mean(),
                    'ambient_std': m_df[f'{method}_ambient_cost'].std(),
                    'latent_mean': m_df[f'{method}_latent_effort'].mean(),
                    'latent_std': m_df[f'{method}_latent_effort'].std(),
                })
        summary = pd.DataFrame(rows_list)

        print('\\begin{table}[t]')
        print('\\centering')
        print('\\caption{Effort Evaluation: Standard OT vs Causal OT (mean $\\pm$ std across 5-fold CV)}')
        print('\\label{tab:recourse_effort_cv}')
        print('\\begin{tabular}{lcccc}')
        print('\\toprule')
        print('Dataset & $\\kappa$ & Method & Ambient Cost & Latent Effort\\\\')
        print('\\midrule')
        for _, row in summary.iterrows():
            d = row['dataset']
            k = f"{row['kappa_mean']:.2f} $\\pm$ {row['kappa_std']:.2f}"
            method = row['method']
#           v = f"{row['validity_mean']:.1f} $\\pm$ {row['validity_std']:.1f}"
            amb = f"{row['ambient_mean']:.3f} $\\pm$ {row['ambient_std']:.3f}"
            lat = f"{row['latent_mean']:.3f} $\\pm$ {row['latent_std']:.3f}"
            print(f'{d} & {k} & {method} & {amb} & {lat} \\\\')
#            print(f'{d} & {k} & {method} & {v} & {amb} & {lat} \\\\')
            if method == 'Causal OT':
                print('\\midrule')
        print('\\bottomrule')
        print('\\end{tabular}')
        print('\\end{table}')
    else:
        print('\\begin{table}[t]')
        print('\\centering')
        print('\\caption{Effort Evaluation: Standard OT vs Causal OT}')
        print('\\label{tab:recourse_effort}')
        print('\\begin{tabular}{lcccc}')
        print('\\toprule')
        print('Dataset & $\\kappa$ & Method & Validity (\\%) & Ambient Cost & Latent Effort\\\\')
        print('\\midrule')
        for _, row in df15.iterrows():
            d = row['dataset']
            k = row['kappa']
            for method in ['Standard OT', 'Causal OT']:
#                v = row[f'{method}_validity']
                amb = row[f'{method}_ambient_cost']
                lat = row[f'{method}_latent_effort']
                print(f'{d} & {k:.2f} & {method} & {amb:.3f} & {lat:.3f} \\\\')
            print('\\midrule')
        print('\\bottomrule')
        print('\\end{tabular}')
        print('\\end{table}')
else:
    print('No C2/C3 results to render.')


---
## C3 — Causal OT on Real / Realistic Data

Part A (ecoli70, rendered in the C2 table above): Standard OT vs Causal OT on data simulated from the ecoli70 Bayesian network (46 nodes), 5-fold CV.
Part B: conjugacy deviation between the ambient (X) and latent (U, PCA) spaces across lsac / student / credit_default with 10-fold CV.


In [ ]:
# C3 — Part B: conjugacy deviation on real data (PCA latent space)
df_c3real = load_csv('c3_causality_real')

if not df_c3real.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(df_c3real))
    ax.bar(x, df_c3real['diff_mean'], yerr=df_c3real['diff_std'], capsize=4,
           color='#1f77b4', alpha=0.85)
    for xi, r in zip(x, df_c3real.itertuples()):
        ax.text(xi, r.diff_mean, f"EV={r.explained_var_ratio:.2f}",
                ha='center', va='bottom', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(df_c3real['dataset'])
    ax.set_ylabel(r'$\|\hat{T}_X - M\,\hat{T}_U\|^2$', size=15)
    ax.set_title('C3 — Ambient vs latent transport deviation (real data)', size=20)
    ax.grid(True, axis='y', alpha=0.3)
    fig.tight_layout()
    savefig(fig, 'c3_real_conjugacy')
    plt.close(fig)
else:
    print('C3 real-data results not found — run c3_causality_real.py first')
